# Newspaper portrayal analysis of Israel and Palestine
In this project, we leverage a pre-trained word embedding model, specifically RoBERTa, on multiples newspapers' corpora. We create these corpora by scraping websites of different sources. If you would like to see how we scraped, please check out the [github repository](https://github.com/McGill-AI-Lab/news-bias-model) for the project

#### Note:
In the following jupyter notebook, we had to delete some of the cell outputs due to either them being too large, or some copyright constraints. Please know that we ran all of the code in the notebook.

### Peter's proposed solution:
- Build BERT masked word string with article, at the end add "Palestine is _" and "Isreal is _", evaluate with list of all words relative to good and bad
- Accumulate the results per article for this newspaper institution's overall score



## High-level Pipeline Method

In [20]:
# Import requirement libraries
# import torch
# from transformers import pipeline

# pipeline = pipeline(
#     task="fill-mask",
#     model="FacebookAI/roberta-base",
#     dtype=torch.float16,
#     device=0
# )
# pipeline("Plants create <mask> through a process known as photosynthesis.")


## Low-level AutoModel Method that works for Mac Users

In [38]:
#Import Libraries
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
from typing import List, Tuple

In [ ]:
def retrieve_results(article, model, tokenizer, device) -> Tuple[List[str], List[float]]:
    text = f"We read the following sentence and answer if Palestine is good or bad:" \
    f"{article}" \
    "We answer: Palestine is <mask>."
    
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_id = tokenizer.mask_token_id
    pos = (inputs["input_ids"] == mask_id).nonzero(as_tuple=False)[0]
    b, i = pos[0].item(), pos[1].item()

    k = 10
    scores, token_ids = torch.topk(logits[b, i], k)

    # print(type(scores))
    # print(scores)
    # print(sum(scores))

    probs = torch.softmax(logits[b, i], dim=-1)[token_ids]

    cands = [tokenizer.decode(t).strip() for t in token_ids.tolist()]

    return cands, probs.tolist()    

In [44]:
model_id = "FacebookAI/roberta-base"

# Pick the best available device on a Mac: MPS > CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32  # safer on MPS & CPU
).to(device)

article = "A random sentence."
word, prob = retrieve_results(article, model, tokenizer, device)

for c, p in zip(word, prob):
        print(f"{c}: {p:.4f}")


<class 'torch.Tensor'>
tensor([17.9485, 17.8762, 14.0698, 13.4306, 13.2713, 13.1769, 12.3632, 12.3186,
        12.3109, 12.2105], device='mps:0')
tensor(138.9766, device='mps:0')
good: 0.4876
bad: 0.4536
great: 0.0101
terrible: 0.0053
evil: 0.0045
better: 0.0041
Good: 0.0018
horrible: 0.0017
fine: 0.0017
nice: 0.0016
